# Assignment 1: Comprehensive Performance Analysis of CUDA Matrix Multiplication

This notebook uses the `nvcc4jupyter` plugin you mentioned to write, compile, and run CUDA code directly in the cells. We can also use standard Python cells below it to plot the results!

In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

In [ ]:
get_ipython().run_line_magic("reload_ext", "nvcc4jupyter")

from pathlib import Path
import re
from nvcc4jupyter.plugin import NVCCPlugin

if not hasattr(NVCCPlugin, "_original_compile_for_timing_patch"):
    NVCCPlugin._original_compile_for_timing_patch = NVCCPlugin._compile
original_compile = NVCCPlugin._original_compile_for_timing_patch
utils_path = Path("utils.cuh").resolve()
timings_path = Path("gemm_timings.csv").resolve()
timing_pattern = re.compile(
    r"  float elapsed_time;\s+"
    r"cudaEventElapsedTime\(&elapsed_time, beg, end\);\s+"
    r"elapsed_time\s*/=\s*1000\.0;\s*// Convert to seconds\s+"
    r"long flops = 2L \* M \* N \* K;\s+"
    r"float gflops = \(repeat_times \* flops \* 1e-9\) / elapsed_time;"
)
timing_replacement = (
    "  float elapsed_ms = 0.0f;\n"
    "  cudaEventElapsedTime(&elapsed_ms, beg, end);\n"
    "  const double elapsed_seconds = elapsed_ms / 1000.0;\n\n"
    "  const double flops = 2.0 * M * N * K;\n"
    "  const double gflops = (repeat_times * flops * 1e-9) / elapsed_seconds;"
)

def compile_with_utils(self, group_name, executable_fname="cuda_exec.out"):
    group_dir = Path(self.workdir) / group_name
    shared_dir = Path(self.workdir) / "shared"
    shared_dir.mkdir(exist_ok=True)
    (shared_dir / "utils.cuh").write_bytes(utils_path.read_bytes())
    patched_sources = 0

    for source_path in group_dir.glob("*.cu"):
        source = source_path.read_text()
        source, replacements = timing_pattern.subn(timing_replacement, source, count=1)
        if replacements:
            patched_sources += 1
        source = source.replace(
            'printf("Elapsed time: %.6f seconds\\n", elapsed_time);',
            'printf("Elapsed time: %.6f seconds\\n", elapsed_seconds);',
            1,
        )
        match = re.search(
            r'printf\("kernel_%s: %.1f GFLOPS\\n", "([^"]+)", gflops\);',
            source,
        )
        if match and "gemm_timings.csv" not in source:
            kernel_name = match.group(1)
            elapsed_line = '  printf("Elapsed time: %.6f seconds\\n", elapsed_seconds);\n'
            elapsed_code = "" if elapsed_line in source else elapsed_line
            timing_code = (
                f'{match.group(0)}\n'
                f'{elapsed_code}'
                f'  FILE *timing_file = fopen("{timings_path}", "a");\n'
                f'  if (timing_file != nullptr) {{\n'
                f'    fprintf(timing_file, "{kernel_name},%.9f,%.3f\\n", elapsed_seconds, gflops);\n'
                f'    fclose(timing_file);\n'
                f'  }}'
            )
            source = source.replace(match.group(0), timing_code, 1)
        source_path.write_text(source)

    if patched_sources == 0:
        print(f"Timing patch matched no CUDA source files in {group_name}.")


    print(f"Patched {patched_sources} CUDA source file(s) in {group_name}.")
    return original_compile(self, group_name, executable_fname)

NVCCPlugin._compile = compile_with_utils
NVCCPlugin._utils_cuh_timing_patch = True

In [ ]:
import os
from pathlib import Path

os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
Path('gemm_timings.csv').write_text('Kernel,ElapsedSeconds,GFLOPS\n')

In [ ]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

// 1. Naive Matrix Multiplication
__global__ void naive_matmul(float* A, float* B, float* C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    
    if(row < M && col < N) {
        float sum = 0.0f;
        for(int i = 0; i < K; ++i) {
            sum += A[row * K + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    cout << "Naive Matmul Compiled and Ready!" << endl;
    // We will add the memory allocation and kernel launches here
    return 0;
}

### Utility Header
Run this cell to write the `utils.cuh` header containing our matrix initialization routines.

In [ ]:
%%writefile utils.cuh
#pragma once
#include <iostream>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <sys/time.h>

inline void randomize_matrix(float *mat, int N) {
  struct timeval time {};
  gettimeofday(&time, nullptr);
  srand(time.tv_usec);
  for (int i = 0; i < N; i++) {
    float tmp = (float)(rand() % 5) + 0.01 * (rand() % 5);
    tmp = (rand() % 2 == 0) ? tmp : tmp * (-1.);
    mat[i] = tmp;
  }
}


### Kernel 1: Naive Matrix Multiplication

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

/*

Matrix sizes:
MxK * KxN = MxN

*/

__global__ void sgemm_naive(int M, int N, int K, float alpha, const float *A,
                            const float *B, float beta, float *C) {
  const uint x = blockIdx.x * blockDim.x + threadIdx.x;
  const uint y = blockIdx.y * blockDim.y + threadIdx.y;

  // if statement is necessary to make things work under tile quantization
  if (x < M && y < N) {
    float tmp = 0.0;
    for (int i = 0; i < K; ++i) {
      tmp += A[x * K + i] * B[i * N + y];
    }
    // C = α*(A@B)+β*C
    C[x * N + y] = alpha * tmp + beta * C[x * N + y];
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  #define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))
  dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32));
  dim3 blockDim(32, 32);
  sgemm_naive<<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "1_naive", gflops);
  printf("Elapsed time: %.6f seconds\n", elapsed_time);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 2: Global Memory Coalescing

In this approach we will use the shared memory to improve 


So shared memory is the memory which is accessed by all threads in a block (not a warp).

Much faster than loading from the global memory.

Here what happens for a thread block of size 16x16 each thread will load a element from the global memory into shared memory. Then, all threads will synchronize and perform the multiplication using the data in shared memory. This reduces the number of global memory accesses and improves performance.

SO a 16x16 thread block load 256 elemnts and then all threads perform using this data to compute total of 256 multiplications so that we get a total of 256 multiplications using only 16 global memory accesses. (256x16=4096 multiplications which are required to calculate value for one element of output matrix).
In one operation 


remember the number of blocks = 4096/256 = 16x16 thread blocks.
each block runs simultaneously 

each block calculates the value of a 16x16 matrix

but since we need to do 4096 calculations for one element so what will happen after one block finishes its computation, it will have to load the next 256 elements from global memory into shared memory and repeat the process until all 4096 multiplications are completed for that element.

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

template <const uint BLOCKSIZE>
__global__ void sgemm_global_mem_coalesce(int M, int N, int K, float alpha,
                                          const float *A, const float *B,
                                          float beta, float *C) {
  const int cRow = blockIdx.x * BLOCKSIZE + (threadIdx.x / BLOCKSIZE);
  const int cCol = blockIdx.y * BLOCKSIZE + (threadIdx.x % BLOCKSIZE);

  // if statement is necessary to make things work under tile quantization
  if (cRow < M && cCol < N) {
    float tmp = 0.0;
    for (int i = 0; i < K; ++i) {
      tmp += A[cRow * K + i] * B[i * N + cCol];
    }
    C[cRow * N + cCol] = alpha * tmp + beta * C[cRow * N + cCol];
  }
}


#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))
void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32));
  dim3 blockDim(32 * 32);
  sgemm_global_mem_coalesce<32><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "2_kernel_global_mem_coalesce", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 3: Shared Memory Cache-Blocking

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

template <const int BLOCKSIZE>
__global__ void sgemm_shared_mem_block(int M, int N, int K, float alpha,
                                       const float *A, const float *B,
                                       float beta, float *C) {
  // the output block that we want to compute in this threadblock
  const uint cRow = blockIdx.x;
  const uint cCol = blockIdx.y;

  // allocate buffer for current block in fast shared mem
  // shared mem is shared between all threads in a block
  __shared__ float As[BLOCKSIZE * BLOCKSIZE];
  __shared__ float Bs[BLOCKSIZE * BLOCKSIZE];

  // the inner row & col that we're accessing in this thread
  const uint threadCol = threadIdx.x % BLOCKSIZE;
  const uint threadRow = threadIdx.x / BLOCKSIZE;

  // advance pointers to the starting positions
  A += cRow * BLOCKSIZE * K;                    // row=cRow, col=0
  B += cCol * BLOCKSIZE;                        // row=0, col=cCol
  C += cRow * BLOCKSIZE * N + cCol * BLOCKSIZE; // row=cRow, col=cCol

  float tmp = 0.0;
  for (int bkIdx = 0; bkIdx < K; bkIdx += BLOCKSIZE) {
    // Have each thread load one of the elements in A & B
    // Make the threadCol (=threadIdx.x) the consecutive index
    // to allow global memory access coalescing
    As[threadRow * BLOCKSIZE + threadCol] = A[threadRow * K + threadCol];
    Bs[threadRow * BLOCKSIZE + threadCol] = B[threadRow * N + threadCol];

    // block threads in this block until cache is fully populated
    __syncthreads();
    A += BLOCKSIZE;
    B += BLOCKSIZE * N;

    // execute the dotproduct on the currently cached block
    for (int dotIdx = 0; dotIdx < BLOCKSIZE; ++dotIdx) {
      tmp += As[threadRow * BLOCKSIZE + dotIdx] *
             Bs[dotIdx * BLOCKSIZE + threadCol];
    }
    // need to sync again at the end, to avoid faster threads
    // fetching the next block into the cache before slower threads are done
    __syncthreads();
  }
  C[threadRow * N + threadCol] =
      alpha * tmp + beta * C[threadRow * N + threadCol];
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32));
  dim3 blockDim(32 * 32);
  cudaFuncSetAttribute(sgemm_shared_mem_block<32>, cudaFuncAttributePreferredSharedMemoryCarveout, cudaSharedmemCarveoutMaxShared);
  sgemm_shared_mem_block<32><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "3_kernel_shared_mem_blocking", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 4: 1D Blocktiling

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

template <const int BM, const int BN, const int BK, const int TM>
__global__ void sgemm1DBlocktiling(int M, int N, int K, float alpha,
                                   const float *A, const float *B, float beta,
                                   float *C) {
  // If we flip x and y here we get ~30% less performance for large matrices.
  // The current, 30% faster configuration ensures that blocks with sequential
  // blockIDs access columns of B sequentially, while sharing the same row of A.
  // The slower configuration would share columns of A, but access into B would
  // be non-sequential. So the faster configuration has better spatial locality
  // and hence a greater L2 hit rate.
  const uint cRow = blockIdx.y;
  const uint cCol = blockIdx.x;

  // each warp will calculate 32*TM elements, with 32 being the columnar dim.
  const int threadCol = threadIdx.x % BN;
  const int threadRow = threadIdx.x / BN;

  // allocate space for the current blocktile in SMEM
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  // Move blocktile to beginning of A's row and B's column
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;

  // todo: adjust this to each thread to load multiple entries and
  // better exploit the cache sizes
  assert(BM * BK == blockDim.x);
  assert(BN * BK == blockDim.x);
  const uint innerColA = threadIdx.x % BK; // warp-level GMEM coalescing
  const uint innerRowA = threadIdx.x / BK;
  const uint innerColB = threadIdx.x % BN; // warp-level GMEM coalescing
  const uint innerRowB = threadIdx.x / BN;

  // allocate thread-local cache for results in registerfile
  float threadResults[TM] = {0.0};

  // outer loop over block tiles
  for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
    // populate the SMEM caches
    As[innerRowA * BK + innerColA] = A[innerRowA * K + innerColA];
    Bs[innerRowB * BN + innerColB] = B[innerRowB * N + innerColB];
    __syncthreads();

    // advance blocktile
    A += BK;
    B += BK * N;

    // calculate per-thread results
    for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
      // we make the dotproduct loop the outside loop, which facilitates
      // reuse of the Bs entry, which we can cache in a tmp var.
      float tmpB = Bs[dotIdx * BN + threadCol];
      for (uint resIdx = 0; resIdx < TM; ++resIdx) {
        threadResults[resIdx] +=
            As[(threadRow * TM + resIdx) * BK + dotIdx] * tmpB;
      }
    }
    __syncthreads();
  }

  // write out the results
  for (uint resIdx = 0; resIdx < TM; ++resIdx) {
    C[(threadRow * TM + resIdx) * N + threadCol] =
        alpha * threadResults[resIdx] +
        beta * C[(threadRow * TM + resIdx) * N + threadCol];
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  const uint BM = 64;
  const uint BN = 64;
  const uint BK = 8;
  const uint TM = 8;
  dim3 gridDim(CEIL_DIV(N, BN), CEIL_DIV(M, BM));
  dim3 blockDim((BM * BN) / TM);
  sgemm1DBlocktiling<BM, BN, BK, TM><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "4_kernel_1D_blocktiling", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 5: 2D Blocktiling

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

template <const int BM, const int BN, const int BK, const int TM, const int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN), 1)
    sgemm2DBlocktiling(int M, int N, int K, float alpha, const float *A,
                       const float *B, float beta, float *C) {
  const uint cRow = blockIdx.y;
  const uint cCol = blockIdx.x;

  const uint totalResultsBlocktile = BM * BN;
  // A thread is responsible for calculating TM*TN elements in the blocktile
  const uint numThreadsBlocktile = totalResultsBlocktile / (TM * TN);

  // ResultsPerBlock / ResultsPerThread == ThreadsPerBlock
  assert(numThreadsBlocktile == blockDim.x);

  // BN/TN are the number of threads to span a column
  const int threadCol = threadIdx.x % (BN / TN);
  const int threadRow = threadIdx.x / (BN / TN);

  // allocate space for the current blocktile in smem
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  // Move blocktile to beginning of A's row and B's column
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;

  // calculating the indices that this thread will load into SMEM
  const uint innerRowA = threadIdx.x / BK;
  const uint innerColA = threadIdx.x % BK;
  // calculates the number of rows of As that are being loaded in a single step
  // by a single block
  const uint strideA = numThreadsBlocktile / BK;
  const uint innerRowB = threadIdx.x / BN;
  const uint innerColB = threadIdx.x % BN;
  // for both As and Bs we want each load to span the full column-width, for
  // better GMEM coalescing (as opposed to spanning full row-width and iterating
  // across columns)
  const uint strideB = numThreadsBlocktile / BN;

  // allocate thread-local cache for results in registerfile
  float threadResults[TM * TN] = {0.0};
  // register caches for As and Bs
  float regM[TM] = {0.0};
  float regN[TN] = {0.0};

  // outer-most loop over block tiles
  for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
    // populate the SMEM caches
    for (uint loadOffset = 0; loadOffset < BM; loadOffset += strideA) {
      As[(innerRowA + loadOffset) * BK + innerColA] =
          A[(innerRowA + loadOffset) * K + innerColA];
    }
    for (uint loadOffset = 0; loadOffset < BK; loadOffset += strideB) {
      Bs[(innerRowB + loadOffset) * BN + innerColB] =
          B[(innerRowB + loadOffset) * N + innerColB];
    }
    __syncthreads();

    // advance blocktile
    A += BK;     // move BK columns to right
    B += BK * N; // move BK rows down

    // calculate per-thread results
    for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
      // block into registers
      for (uint i = 0; i < TM; ++i) {
        regM[i] = As[(threadRow * TM + i) * BK + dotIdx];
      }
      for (uint i = 0; i < TN; ++i) {
        regN[i] = Bs[dotIdx * BN + threadCol * TN + i];
      }
      for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
        for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
          threadResults[resIdxM * TN + resIdxN] +=
              regM[resIdxM] * regN[resIdxN];
        }
      }
    }
    __syncthreads();
  }

  // write out the results
  for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
    for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
      C[(threadRow * TM + resIdxM) * N + threadCol * TN + resIdxN] =
          alpha * threadResults[resIdxM * TN + resIdxN] +
          beta * C[(threadRow * TM + resIdxM) * N + threadCol * TN + resIdxN];
    }
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  const uint BK = 8;
  const uint TM = 8;
  const uint TN = 8;
  const uint BM = 128;
  const uint BN = 128;
  dim3 gridDim(CEIL_DIV(N, BN), CEIL_DIV(M, BM));
  dim3 blockDim((BM * BN) / (TM * TN));
  sgemm2DBlocktiling<BM, BN, BK, TM, TN><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "5_kernel_2D_blocktiling", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 6: Vectorized SMEM and GMEM Accesses

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

template <const int BM, const int BN, const int BK, const int TM, const int TN>
__global__ void sgemmVectorize(int M, int N, int K, float alpha, float *A,
                               float *B, float beta, float *C) {
  const uint cRow = blockIdx.y;
  const uint cCol = blockIdx.x;

  // BN/TN are the number of threads to span a column
  const int threadCol = threadIdx.x % (BN / TN);
  const int threadRow = threadIdx.x / (BN / TN);

  // allocate space for the current blocktile in smem
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  // Move blocktile to beginning of A's row and B's column
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;

  // calculating the indices that this thread will load into SMEM
  // we'll load 128bit / 32bit = 4 elements per thread at each step
  const uint innerRowA = threadIdx.x / (BK / 4);
  const uint innerColA = threadIdx.x % (BK / 4);
  const uint innerRowB = threadIdx.x / (BN / 4);
  const uint innerColB = threadIdx.x % (BN / 4);

  // allocate thread-local cache for results in registerfile
  float threadResults[TM * TN] = {0.0};
  float regM[TM] = {0.0};
  float regN[TN] = {0.0};

  // outer-most loop over block tiles
  for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
    // populate the SMEM caches
    // transpose A while loading it
    float4 tmp =
        reinterpret_cast<float4 *>(&A[innerRowA * K + innerColA * 4])[0];
    As[(innerColA * 4 + 0) * BM + innerRowA] = tmp.x;
    As[(innerColA * 4 + 1) * BM + innerRowA] = tmp.y;
    As[(innerColA * 4 + 2) * BM + innerRowA] = tmp.z;
    As[(innerColA * 4 + 3) * BM + innerRowA] = tmp.w;

    reinterpret_cast<float4 *>(&Bs[innerRowB * BN + innerColB * 4])[0] =
        reinterpret_cast<float4 *>(&B[innerRowB * N + innerColB * 4])[0];
    __syncthreads();

    // advance blocktile
    A += BK;     // move BK columns to right
    B += BK * N; // move BK rows down

    // calculate per-thread results
    for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
      // block into registers
      for (uint i = 0; i < TM; ++i) {
        regM[i] = As[dotIdx * BM + threadRow * TM + i];
      }
      for (uint i = 0; i < TN; ++i) {
        regN[i] = Bs[dotIdx * BN + threadCol * TN + i];
      }
      for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
        for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
          threadResults[resIdxM * TN + resIdxN] +=
              regM[resIdxM] * regN[resIdxN];
        }
      }
    }
    __syncthreads();
  }

  // write out the results
  for (uint resIdxM = 0; resIdxM < TM; resIdxM += 1) {
    for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
      // load C vector into registers
      float4 tmp = reinterpret_cast<float4 *>(
          &C[(threadRow * TM + resIdxM) * N + threadCol * TN + resIdxN])[0];
      // perform GEMM update in reg
      tmp.x = alpha * threadResults[resIdxM * TN + resIdxN] + beta * tmp.x;
      tmp.y = alpha * threadResults[resIdxM * TN + resIdxN + 1] + beta * tmp.y;
      tmp.z = alpha * threadResults[resIdxM * TN + resIdxN + 2] + beta * tmp.z;
      tmp.w = alpha * threadResults[resIdxM * TN + resIdxN + 3] + beta * tmp.w;
      // write back
      reinterpret_cast<float4 *>(
          &C[(threadRow * TM + resIdxM) * N + threadCol * TN + resIdxN])[0] =
          tmp;
    }
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  const uint BK = 8;
  const uint TM = 8;
  const uint TN = 8;
  const uint BM = 128;
  const uint BN = 128;
  dim3 gridDim(CEIL_DIV(N, BN), CEIL_DIV(M, BM));
  dim3 blockDim((BM * BN) / (TM * TN));
  sgemmVectorize<BM, BN, BK, TM, TN><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "6_kernel_vectorize", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 9: Autotuning

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))
const int K9_NUM_THREADS = 256;

template <const int BM, const int BN, const int BK, const int TM, const int TN>
__global__ void __launch_bounds__(K9_NUM_THREADS)
    sgemmAutotuned(int M, int N, int K, float alpha, float *A, float *B,
                   float beta, float *C) {
  const uint cRow = blockIdx.y;
  const uint cCol = blockIdx.x;

  // size of warptile
  constexpr int WM = TM * 16;
  constexpr int WN = TN * 16;
  // iterations of warptile
  constexpr int WMITER = CEIL_DIV(BM, WM);
  constexpr int WNITER = CEIL_DIV(BN, WN);

  // Placement of the thread in the warptile
  const int threadCol = threadIdx.x % (WN / TN);
  const int threadRow = threadIdx.x / (WN / TN);

  // allocate space for the current blocktile in smem
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  // Move blocktile to beginning of A's row and B's column
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;

  // calculating the indices that this thread will load into SMEM
  // we'll load 128bit / 32bit = 4 elements per thread at each step
  const uint innerRowA = threadIdx.x / (BK / 4);
  const uint innerColA = threadIdx.x % (BK / 4);
  constexpr uint rowStrideA = (K9_NUM_THREADS * 4) / BK;
  const uint innerRowB = threadIdx.x / (BN / 4);
  const uint innerColB = threadIdx.x % (BN / 4);
  constexpr uint rowStrideB = K9_NUM_THREADS / (BN / 4);

  // allocate thread-local cache for results in registerfile
  float threadResults[WMITER * WNITER * TM * TN] = {0.0};
  float regM[TM] = {0.0};
  float regN[TN] = {0.0};

  // outer-most loop over block tiles
  for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
    // populate the SMEM caches
    for (uint offset = 0; offset + rowStrideA <= BM; offset += rowStrideA) {
      float4 tmp = reinterpret_cast<float4 *>(
          &A[(innerRowA + offset) * K + innerColA * 4])[0];
      // transpose A while storing it
      As[(innerColA * 4 + 0) * BM + innerRowA + offset] = tmp.x;
      As[(innerColA * 4 + 1) * BM + innerRowA + offset] = tmp.y;
      As[(innerColA * 4 + 2) * BM + innerRowA + offset] = tmp.z;
      As[(innerColA * 4 + 3) * BM + innerRowA + offset] = tmp.w;
    }

    for (uint offset = 0; offset + rowStrideB <= BK; offset += rowStrideB) {
      reinterpret_cast<float4 *>(
          &Bs[(innerRowB + offset) * BN + innerColB * 4])[0] =
          reinterpret_cast<float4 *>(
              &B[(innerRowB + offset) * N + innerColB * 4])[0];
    }
    __syncthreads();

    for (uint wmIdx = 0; wmIdx < WMITER; ++wmIdx) {
      for (uint wnIdx = 0; wnIdx < WNITER; ++wnIdx) {
        // calculate per-thread results
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
          // block into registers
          for (uint i = 0; i < TM; ++i) {
            regM[i] = As[dotIdx * BM + (wmIdx * WM) + threadRow * TM + i];
          }
          for (uint i = 0; i < TN; ++i) {
            regN[i] = Bs[dotIdx * BN + (wnIdx * WN) + threadCol * TN + i];
          }
          for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
            for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
              threadResults[(wmIdx * TM + resIdxM) * (WNITER * TN) +
                            wnIdx * TN + resIdxN] +=
                  regM[resIdxM] * regN[resIdxN];
            }
          }
        }
      }
    }
    __syncthreads();
    // advance blocktile
    A += BK;     // move BK columns to right
    B += BK * N; // move BK rows down
  }

  // write out the results
  for (uint wmIdx = 0; wmIdx < WMITER; ++wmIdx) {
    for (uint wnIdx = 0; wnIdx < WNITER; ++wnIdx) {
      float *C_interim = C + (wmIdx * WM * N) + (wnIdx * WN);
      for (uint resIdxM = 0; resIdxM < TM; resIdxM += 1) {
        for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
          // load C vector into registers
          float4 tmp = reinterpret_cast<float4 *>(
              &C_interim[(threadRow * TM + resIdxM) * N + threadCol * TN +
                         resIdxN])[0];
          // perform GEMM update in reg
          const int i =
              (wmIdx * TM + resIdxM) * (WNITER * TN) + wnIdx * TN + resIdxN;
          tmp.x = alpha * threadResults[i + 0] + beta * tmp.x;
          tmp.y = alpha * threadResults[i + 1] + beta * tmp.y;
          tmp.z = alpha * threadResults[i + 2] + beta * tmp.z;
          tmp.w = alpha * threadResults[i + 3] + beta * tmp.w;
          // write back
          reinterpret_cast<float4 *>(&C_interim[(threadRow * TM + resIdxM) * N +
                                                threadCol * TN + resIdxN])[0] =
              tmp;
        }
      }
    }
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  const uint K9_BK = 16;
  const uint K9_TM = 8;
  const uint K9_TN = 8;
  const uint K9_BM = 128;
  const uint K9_BN = 128;
  dim3 blockDim(K9_NUM_THREADS);
  dim3 gridDim(CEIL_DIV(N, K9_BN), CEIL_DIV(M, K9_BM));
  sgemmAutotuned<K9_BM, K9_BN, K9_BK, K9_TM, K9_TN><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "9_kernel_autotuned", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Kernel 10: Warptiling

In [ ]:
%%cuda
#include "utils.cuh"

#pragma once

#include <algorithm>
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cublas_v2.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))
const int WARPSIZE = 32; // warpSize is not constexpr

namespace wt {
template <const int BM, const int BN, const int BK, const int rowStrideA,
          const int rowStrideB>
__device__ void loadFromGmem(int N, int K, const float *A, const float *B,
                             float *As, float *Bs, int innerRowA, int innerColA,
                             int innerRowB, int innerColB) {
  for (uint offset = 0; offset + rowStrideA <= BM; offset += rowStrideA) {
    const float4 tmp = reinterpret_cast<const float4 *>(
        &A[(innerRowA + offset) * K + innerColA * 4])[0];
    // float4 tmp;
    // asm("ld.global.nc.v4.f32 {%0, %1, %2, %3}, [%4];"
    //     : "=f"(tmp.x), "=f"(tmp.y), "=f"(tmp.z), "=f"(tmp.w)
    //     : "l"(&A[(innerRowA + offset) * K + innerColA * 4]));
    As[(innerColA * 4 + 0) * BM + innerRowA + offset] = tmp.x;
    As[(innerColA * 4 + 1) * BM + innerRowA + offset] = tmp.y;
    As[(innerColA * 4 + 2) * BM + innerRowA + offset] = tmp.z;
    As[(innerColA * 4 + 3) * BM + innerRowA + offset] = tmp.w;
  }

  for (uint offset = 0; offset + rowStrideB <= BK; offset += rowStrideB) {
    reinterpret_cast<float4 *>(
        &Bs[(innerRowB + offset) * BN + innerColB * 4])[0] =
        reinterpret_cast<const float4 *>(
            &B[(innerRowB + offset) * N + innerColB * 4])[0];
    // asm("ld.global.v4.f32 {%0, %1, %2, %3}, [%4];"
    //     : "=f"(Bs[(innerRowB + offset) * BN + innerColB * 4 + 0]),
    //       "=f"(Bs[(innerRowB + offset) * BN + innerColB * 4 + 1]),
    //       "=f"(Bs[(innerRowB + offset) * BN + innerColB * 4 + 2]),
    //       "=f"(Bs[(innerRowB + offset) * BN + innerColB * 4 + 3])
    //     : "l"(&B[(innerRowB + offset) * N + innerColB * 4]));
  }
}

template <const int BM, const int BN, const int BK, const int WM, const int WN,
          const int WMITER, const int WNITER, const int WSUBM, const int WSUBN,
          const int TM, const int TN>
__device__ void
processFromSmem(float *regM, float *regN, float *threadResults, const float *As,
                const float *Bs, const uint warpRow, const uint warpCol,
                const uint threadRowInWarp, const uint threadColInWarp) {
  for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
    // populate registers for whole warptile
    for (uint wSubRowIdx = 0; wSubRowIdx < WMITER; ++wSubRowIdx) {
      for (uint i = 0; i < TM; ++i) {
        regM[wSubRowIdx * TM + i] =
            As[(dotIdx * BM) + warpRow * WM + wSubRowIdx * WSUBM +
               threadRowInWarp * TM + i];
      }
    }
    for (uint wSubColIdx = 0; wSubColIdx < WNITER; ++wSubColIdx) {
      for (uint i = 0; i < TN; ++i) {
        regN[wSubColIdx * TN + i] =
            Bs[(dotIdx * BN) + warpCol * WN + wSubColIdx * WSUBN +
               threadColInWarp * TN + i];
      }
    }

    // execute warptile matmul
    for (uint wSubRowIdx = 0; wSubRowIdx < WMITER; ++wSubRowIdx) {
      for (uint wSubColIdx = 0; wSubColIdx < WNITER; ++wSubColIdx) {
        // calculate per-thread results
        for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
          for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
            threadResults[(wSubRowIdx * TM + resIdxM) * (WNITER * TN) +
                          (wSubColIdx * TN) + resIdxN] +=
                regM[wSubRowIdx * TM + resIdxM] *
                regN[wSubColIdx * TN + resIdxN];
          }
        }
      }
    }
  }
}

} // namespace wt

/*
 * @tparam BM The threadblock size for M dimension SMEM caching.
 * @tparam BN The threadblock size for N dimension SMEM caching.
 * @tparam BK The threadblock size for K dimension SMEM caching.
 * @tparam WM M dim of continuous tile computed by each warp
 * @tparam WN N dim of continuous tile computed by each warp
 * @tparam WMITER The number of subwarp tiling steps in M dimension.
 * @tparam WNITER The number of subwarp tiling steps in N dimension.
 * @tparam TM The per-thread tile size for M dimension.
 * @tparam TN The per-thread tile size for N dimension.
 */
template <const int BM, const int BN, const int BK, const int WM, const int WN,
          const int WNITER, const int TM, const int TN, const int NUM_THREADS>
__global__ void __launch_bounds__(NUM_THREADS)
    sgemmWarptiling(int M, int N, int K, float alpha, float *A, float *B,
                    float beta, float *C) {
  const uint cRow = blockIdx.y;
  const uint cCol = blockIdx.x;

  // Placement of the warp in the threadblock tile
  const uint warpIdx = threadIdx.x / WARPSIZE; // the warp this thread is in
  const uint warpCol = warpIdx % (BN / WN);
  const uint warpRow = warpIdx / (BN / WN);

  // size of the warp subtile
  constexpr uint WMITER = (WM * WN) / (WARPSIZE * TM * TN * WNITER);
  constexpr uint WSUBM = WM / WMITER; // 64/2=32
  constexpr uint WSUBN = WN / WNITER; // 32/2=16

  // Placement of the thread in the warp subtile
  const uint threadIdxInWarp = threadIdx.x % WARPSIZE;         // [0, 31]
  const uint threadColInWarp = threadIdxInWarp % (WSUBN / TN); // i%(16/4)
  const uint threadRowInWarp = threadIdxInWarp / (WSUBN / TN); // i/4

  // allocate space for the current blocktile in SMEM
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];

  // Move blocktile to beginning of A's row and B's column
  A += cRow * BM * K;
  B += cCol * BN;
  // Move C_ptr to warp's output tile
  C += (cRow * BM + warpRow * WM) * N + cCol * BN + warpCol * WN;

  // calculating the indices that this thread will load into SMEM
  // we'll load 128bit / 32bit = 4 elements per thread at each step
  const uint innerRowA = threadIdx.x / (BK / 4);
  const uint innerColA = threadIdx.x % (BK / 4);
  constexpr uint rowStrideA = (NUM_THREADS * 4) / BK;
  const uint innerRowB = threadIdx.x / (BN / 4);
  const uint innerColB = threadIdx.x % (BN / 4);
  constexpr uint rowStrideB = NUM_THREADS / (BN / 4);

  // allocate thread-local cache for results in registerfile
  float threadResults[WMITER * TM * WNITER * TN] = {0.0};
  // we cache into registers on the warptile level
  float regM[WMITER * TM] = {0.0};
  float regN[WNITER * TN] = {0.0};

  // outer-most loop over block tiles
  for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
    wt::loadFromGmem<BM, BN, BK, rowStrideA, rowStrideB>(
        N, K, A, B, As, Bs, innerRowA, innerColA, innerRowB, innerColB);
    __syncthreads();
    wt::processFromSmem<BM, BN, BK, WM, WN, WMITER, WNITER, WSUBM, WSUBN, TM,
                        TN>(regM, regN, threadResults, As, Bs, warpRow, warpCol,
                            threadRowInWarp, threadColInWarp);
    A += BK;     // move BK columns to right
    B += BK * N; // move BK rows down
    __syncthreads();
  }

  // write out the results
  for (uint wSubRowIdx = 0; wSubRowIdx < WMITER; ++wSubRowIdx) {
    for (uint wSubColIdx = 0; wSubColIdx < WNITER; ++wSubColIdx) {
      // move C pointer to current warp subtile
      float *C_interim = C + (wSubRowIdx * WSUBM) * N + wSubColIdx * WSUBN;
      for (uint resIdxM = 0; resIdxM < TM; resIdxM += 1) {
        for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
          // load C vector into registers
          float4 tmp = reinterpret_cast<float4 *>(
              &C_interim[(threadRowInWarp * TM + resIdxM) * N +
                         threadColInWarp * TN + resIdxN])[0];
          // perform GEMM update in reg
          const int i = (wSubRowIdx * TM + resIdxM) * (WNITER * TN) +
                        wSubColIdx * TN + resIdxN;
          tmp.x = alpha * threadResults[i + 0] + beta * tmp.x;
          tmp.y = alpha * threadResults[i + 1] + beta * tmp.y;
          tmp.z = alpha * threadResults[i + 2] + beta * tmp.z;
          tmp.w = alpha * threadResults[i + 3] + beta * tmp.w;
          // write back
          reinterpret_cast<float4 *>(
              &C_interim[(threadRowInWarp * TM + resIdxM) * N +
                         threadColInWarp * TN + resIdxN])[0] = tmp;
        }
      }
    }
  }
}


void run_kernel(int M, int N, int K, float alpha, float *A, float *B, float beta, float *C) {
  const uint K10_NUM_THREADS = 128;
  const uint K10_BN = 128;
  const uint K10_BM = 128;
  const uint K10_BK = 16;
  const uint K10_WM = 64;
  const uint K10_WN = 64;
  const uint K10_WNITER = 4;
  const uint K10_TN = 4;
  const uint K10_TM = 8;
  dim3 blockDim(K10_NUM_THREADS);
  dim3 gridDim(CEIL_DIV(N, K10_BN), CEIL_DIV(M, K10_BM));
  sgemmWarptiling<K10_BM, K10_BN, K10_BK, K10_WM, K10_WN, K10_WNITER, K10_TM, K10_TN, K10_NUM_THREADS><<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}



int main() {
  int M = 4096;
  int N = 4096;
  int K = 4096;
  float alpha = 1.0f;
  float beta = 0.0f;

  float *A = (float *)malloc(sizeof(float) * M * K);
  float *B = (float *)malloc(sizeof(float) * K * N);
  float *C = (float *)malloc(sizeof(float) * M * N);

  randomize_matrix(A, M * K);
  randomize_matrix(B, K * N);
  randomize_matrix(C, M * N);

  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  cudaMemcpy(dA, A, sizeof(float) * M * K, cudaMemcpyHostToDevice);
  cudaMemcpy(dB, B, sizeof(float) * K * N, cudaMemcpyHostToDevice);
  cudaMemcpy(dC, C, sizeof(float) * M * N, cudaMemcpyHostToDevice);

  // Warmup
  run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  cudaDeviceSynchronize();

  int repeat_times = 5;
  cudaEvent_t beg, end;
  cudaEventCreate(&beg);
  cudaEventCreate(&end);

  cudaEventRecord(beg);
  for (int j = 0; j < repeat_times; j++) {
    run_kernel(M, N, K, alpha, dA, dB, beta, dC);
  }
  cudaEventRecord(end);
  cudaEventSynchronize(beg);
  cudaEventSynchronize(end);
  
  float elapsed_time;
  cudaEventElapsedTime(&elapsed_time, beg, end);
  elapsed_time /= 1000.0; // Convert to seconds

  long flops = 2L * M * N * K;
  float gflops = (repeat_times * flops * 1e-9) / elapsed_time;

  printf("kernel_%s: %.1f GFLOPS\n", "10_kernel_warptiling", gflops);

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  free(A); free(B); free(C);
  
  return 0;
}


### Nsight Compute Profiling

The profiling workflow writes `prof_kernel.cu` explicitly before compiling it. The `nvcc` warning about architectures below compute capability 7.5 is non-fatal and is suppressed for this experiment.

For the 4096 x 4096 x 4096 SGEMM, the operation count is approximately:

$$\text{FLOPs} = 2MNK = 2 \times 4096^3 = 137.44\ \text{GFLOP}.$$

The benchmark reports:

$$\text{GFLOP/s} = \frac{2MNK}{t_{kernel}(\text{s}) \times 10^9}.$$

Runtime and GFLOP/s are necessary but not sufficient. Following the reference article, inspect arithmetic intensity, achieved DRAM bandwidth, global-load efficiency, L1/L2 hit rates, shared-memory bank conflicts, registers per thread, active warps, occupancy, and warp stall reasons. The optimized kernels should generally improve arithmetic intensity by reusing data in shared memory and registers; equal timings usually indicate that the displayed precision is too low, the timing logger was not persistent, or the GPU is not actually running the intended distinct kernels.


In [ ]:
%%writefile prof_kernel.cu
#include "utils.cuh"
#include <cublas_v2.h>
#include <cuda_runtime.h>
#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

__global__ void sgemm_naive(int M, int N, int K, float alpha, const float *A, const float *B, float beta, float *C) {
  const uint x = blockIdx.x * blockDim.x + threadIdx.x;
  const uint y = blockIdx.y * blockDim.y + threadIdx.y;
  if (x < M && y < N) {
    float tmp = 0.0f;
    for (int i = 0; i < K; ++i) {
      tmp += A[x * K + i] * B[i * N + y];
    }
    C[x * N + y] = alpha * tmp + beta * C[x * N + y];
  }
}

int main() {
  int M = 4096, N = 4096, K = 4096;
  float *dA, *dB, *dC;
  cudaMalloc(&dA, sizeof(float) * M * K);
  cudaMalloc(&dB, sizeof(float) * K * N);
  cudaMalloc(&dC, sizeof(float) * M * N);

  dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32));
  dim3 blockDim(32, 32);
  sgemm_naive<<<gridDim, blockDim>>>(M, N, K, 1.0f, dA, dB, 0.0f, dC);
  cudaDeviceSynchronize();

  cudaFree(dA); cudaFree(dB); cudaFree(dC);
  return 0;
}


In [ ]:
# Compile for the RTX 2080 Ti (compute capability 7.5).
!nvcc -O3 -arch=sm_75 -Wno-deprecated-gpu-targets prof_kernel.cu -o prof_kernel

# Profile without a kernel-name regex first. This confirms Nsight sees the launch.
!ncu --set full --launch-skip 0 --launch-count 1 ./prof_kernel

### Plotting Performance Results
Use the cell below to plot the performance data.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results = pd.read_csv('gemm_timings.csv')
if results.empty:
    raise RuntimeError('No timing data found. Run the CUDA kernel cells first.')

results = results.groupby('Kernel', as_index=False).mean(numeric_only=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
results.plot.barh(x='Kernel', y='ElapsedSeconds', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('CUDA SGEMM Elapsed Time, 4096 x 4096')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Kernel')
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

results.plot.barh(x='Kernel', y='GFLOPS', ax=axes[1], legend=False, color='darkorange')
axes[1].set_title('CUDA SGEMM Throughput, 4096 x 4096')
axes[1].set_xlabel('GFLOP/s')
axes[1].set_ylabel('')
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# Compare all measured kernels with a cuBLAS SGEMM reference.
from pathlib import Path
import re
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

cublas_source = r'''
#include <cstdio>
#include <cublas_v2.h>
#include <cuda_runtime.h>

int main() {
  const int M = 4096, N = 4096, K = 4096, repeats = 5;
  const size_t bytes_a = size_t(M) * K * sizeof(float);
  const size_t bytes_b = size_t(K) * N * sizeof(float);
  const size_t bytes_c = size_t(M) * N * sizeof(float);
  float *A = nullptr, *B = nullptr, *C = nullptr;
  cublasHandle_t handle = nullptr;
  cudaEvent_t start = nullptr, stop = nullptr;
  cudaError_t cuda_status = cudaMalloc(&A, bytes_a);
  if (cuda_status != cudaSuccess) return 1;
  cuda_status = cudaMalloc(&B, bytes_b);
  if (cuda_status != cudaSuccess) return 1;
  cuda_status = cudaMalloc(&C, bytes_c);
  if (cuda_status != cudaSuccess) return 1;
  cudaMemset(A, 0, bytes_a); cudaMemset(B, 0, bytes_b); cudaMemset(C, 0, bytes_c);
  if (cublasCreate(&handle) != CUBLAS_STATUS_SUCCESS) return 1;
  const float alpha = 1.0f, beta = 0.0f;
  if (cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, M, N, K,
                  &alpha, A, M, B, K, &beta, C, M) != CUBLAS_STATUS_SUCCESS) return 1;
  cudaDeviceSynchronize();
  cudaEventCreate(&start); cudaEventCreate(&stop);
  cudaEventRecord(start);
  for (int i = 0; i < repeats; ++i)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, M, N, K,
                &alpha, A, M, B, K, &beta, C, M);
  cudaEventRecord(stop); cudaEventSynchronize(stop);
  float milliseconds = 0.0f;
  cudaEventElapsedTime(&milliseconds, start, stop);
  const double seconds = milliseconds / 1000.0 / repeats;
  const double gflops = 2.0 * M * N * K / (seconds * 1.0e9);
  std::printf("cuBLAS,%.9f,%.3f\\n", seconds, gflops);
  cublasDestroy(handle); cudaEventDestroy(start); cudaEventDestroy(stop);
  cudaFree(A); cudaFree(B); cudaFree(C);
  return 0;
}
'''

csv_candidates = [Path("gemm_timings.csv"), Path("gemm_timings.csv")]
valid_candidates = []
for candidate in csv_candidates:
    if candidate.exists():
        try:
            frame = pd.read_csv(candidate)
            kernel_rows = frame[frame["Kernel"].astype(str).str.match(r"[1-9][0-9]*_")]
            if not kernel_rows.empty:
                valid_candidates.append((candidate, frame))
        except (KeyError, pd.errors.EmptyDataError):
            pass

if not valid_candidates:
    raise RuntimeError("No measured kernel rows found. Run the CUDA kernel cells before this comparison cell.")

csv_path, results = max(valid_candidates, key=lambda item: len(item[1]))
results = results.groupby("Kernel", as_index=False).mean(numeric_only=True)
print(f"Using kernel timings from: {csv_path}")

source_path = Path("cublas_reference.cu")
executable_path = Path("cublas_reference")
cublas_result = None
try:
    source_path.write_text(cublas_source)
    subprocess.run(
        ["nvcc", "-O3", str(source_path), "-lcublas", "-o", str(executable_path)],
        check=True, capture_output=True, text=True,
    )
    output = subprocess.run([f"./{executable_path}"], check=True, capture_output=True, text=True).stdout
    match = re.search(r"cuBLAS,([0-9.eE+-]+),([0-9.eE+-]+)", output)
    if match:
        cublas_result = {"Kernel": "0: cuBLAS", "ElapsedSeconds": float(match.group(1)), "GFLOPS": float(match.group(2))}
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    logged = results[results["Kernel"].astype(str).str.contains("cuBLAS", case=False, na=False)]
    if not logged.empty:
        row = logged.iloc[0]
        cublas_result = {"Kernel": "0: cuBLAS", "ElapsedSeconds": row["ElapsedSeconds"], "GFLOPS": row["GFLOPS"]}
    else:
        print("cuBLAS benchmark could not run, and no logged cuBLAS row was found.")
        if isinstance(error, subprocess.CalledProcessError):
            print(error.stderr[-1000:])

if cublas_result is None:
    raise RuntimeError("A cuBLAS reference is required to calculate relative performance.")

results = pd.concat([results, pd.DataFrame([cublas_result])], ignore_index=True)
cublas_gflops = cublas_result["GFLOPS"]
results["Performance relative to cuBLAS"] = 100.0 * results["GFLOPS"] / cublas_gflops
results = results.sort_values("Kernel")
comparison = results[["Kernel", "GFLOPS", "Performance relative to cuBLAS"]].round(1)
display(comparison)

ax = results.plot.barh(x="Kernel", y="Performance relative to cuBLAS", legend=False, figsize=(10, 6), color="steelblue")
ax.set_xlabel("Performance relative to cuBLAS (%)")
ax.set_title("All CUDA SGEMM kernels relative to cuBLAS: 4096 x 4096 x 4096")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Parameter Sensitivity Analysis

This section measures how block tiles (`BM`, `BN`, `BK`) and per-thread output tiles (`TM`, `TN`) affect SGEMM performance. The CUDA kernel includes boundary checks, so non-tile-aligned dimensions can be tested safely. Run the following cells after enabling a GPU runtime.

In [ ]:
%%writefile parameter_sensitivity.cu
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <vector>

#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  std::fprintf(stderr, "CUDA error: %s\\n", cudaGetErrorString(e)); std::exit(1); \
} } while (0)
#define CEIL_DIV(x, y) (((x) + (y) - 1) / (y))

template<int BM, int BN, int BK, int TM, int TN>
__global__ void tiled_gemm(int M, int N, int K, const float* A,
                           const float* B, float* C) {
  constexpr int THREADS = (BM / TM) * (BN / TN);
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];
  const int tid = threadIdx.x;
  const int thread_row = tid / (BN / TN);
  const int thread_col = tid % (BN / TN);
  float accum[TM * TN] = {0.0f};

  for (int k0 = 0; k0 < K; k0 += BK) {
    for (int i = tid; i < BM * BK; i += THREADS) {
      const int r = i / BK, c = i % BK;
      const int gr = blockIdx.y * BM + r, gc = k0 + c;
      As[i] = (gr < M && gc < K) ? A[gr * K + gc] : 0.0f;
    }
    for (int i = tid; i < BK * BN; i += THREADS) {
      const int r = i / BN, c = i % BN;
      const int gr = k0 + r, gc = blockIdx.x * BN + c;
      Bs[i] = (gr < K && gc < N) ? B[gr * N + gc] : 0.0f;
    }
    __syncthreads();
    for (int k = 0; k < BK; ++k) {
      for (int r = 0; r < TM; ++r) {
        const float a = As[(thread_row * TM + r) * BK + k];
        for (int c = 0; c < TN; ++c)
          accum[r * TN + c] += a * Bs[k * BN + thread_col * TN + c];
      }
    }
    __syncthreads();
  }
  for (int r = 0; r < TM; ++r) for (int c = 0; c < TN; ++c) {
    const int gr = blockIdx.y * BM + thread_row * TM + r;
    const int gc = blockIdx.x * BN + thread_col * TN + c;
    if (gr < M && gc < N) C[gr * N + gc] = accum[r * TN + c];
  }
}

struct Config { int bm, bn, bk, tm, tn; };

bool legal(Config c) {
  const int threads = (c.bm / c.tm) * (c.bn / c.tn);
  const int smem = (c.bm * c.bk + c.bk * c.bn) * 4;
  return c.bm % c.tm == 0 && c.bn % c.tn == 0 && threads >= 32 &&
         threads <= 1024 && threads % 32 == 0 && smem <= 48 * 1024;
}

template<int BM, int BN, int BK, int TM, int TN>
float measure(int M, int N, int K, const float* A, const float* B,
              float* C, int repeats) {
  constexpr int threads = (BM / TM) * (BN / TN);
  dim3 grid(CEIL_DIV(N, BN), CEIL_DIV(M, BM));
  tiled_gemm<BM, BN, BK, TM, TN><<<grid, threads>>>(M, N, K, A, B, C);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());
  cudaEvent_t start, stop;
  CUDA_CHECK(cudaEventCreate(&start));
  CUDA_CHECK(cudaEventCreate(&stop));
  CUDA_CHECK(cudaEventRecord(start));
  for (int i = 0; i < repeats; ++i)
    tiled_gemm<BM, BN, BK, TM, TN><<<grid, threads>>>(M, N, K, A, B, C);
  CUDA_CHECK(cudaEventRecord(stop));
  CUDA_CHECK(cudaEventSynchronize(stop));
  float ms = 0.0f;
  CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
  CUDA_CHECK(cudaEventDestroy(start));
  CUDA_CHECK(cudaEventDestroy(stop));
  return ms / repeats;
}

template<int BM, int BN, int BK, int TM, int TN>
void record_result(const Config& c, int M, int N, int K, const float* dA,
                  const float* dB, float* dC, int repeats, std::ofstream& csv) {
  const float ms = measure<BM, BN, BK, TM, TN>(M, N, K, dA, dB, dC, repeats);
  const double gflops = 2.0 * M * N * K / (ms * 1.0e6);
  const int threads = (BM / TM) * (BN / TN);
  csv << M << ',' << N << ',' << K << ',' << c.bm << ',' << c.bn << ','
      << c.bk << ',' << c.tm << ',' << c.tn << ',' << threads << ','
      << ms << ',' << gflops << std::endl;
  std::cout << "BM=" << c.bm << " BN=" << c.bn << " BK=" << c.bk
            << " TM=" << c.tm << " TN=" << c.tn << " -> "
            << gflops << " GFLOP/s" << std::endl;
}

int main(int argc, char** argv) {
  const int M = argc > 1 ? std::atoi(argv[1]) : 2048;
  const int N = argc > 2 ? std::atoi(argv[2]) : 2048;
  const int K = argc > 3 ? std::atoi(argv[3]) : 2048;
  const int repeats = argc > 4 ? std::atoi(argv[4]) : 10;
  const size_t bytes_a = size_t(M) * K * sizeof(float);
  const size_t bytes_b = size_t(K) * N * sizeof(float);
  const size_t bytes_c = size_t(M) * N * sizeof(float);
  std::vector<float> hA(size_t(M) * K, 0.01f), hB(size_t(K) * N, 0.02f);
  float *dA, *dB, *dC;
  CUDA_CHECK(cudaMalloc(&dA, bytes_a));
  CUDA_CHECK(cudaMalloc(&dB, bytes_b));
  CUDA_CHECK(cudaMalloc(&dC, bytes_c));
  CUDA_CHECK(cudaMemcpy(dA, hA.data(), bytes_a, cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(dB, hB.data(), bytes_b, cudaMemcpyHostToDevice));
  std::ofstream csv("parameter_sweep.csv");
  csv << "M,N,K,BM,BN,BK,TM,TN,Threads,ElapsedMs,GFLOPS" << std::endl;

  // Exclude 1024-thread/high-register configurations that fail on this GPU.
  const std::vector<Config> configs = {
      {64,64,8,4,4}, {64,64,8,8,8}, {64,64,16,8,8},
      {128,64,8,8,4}, {128,64,16,8,4},
      {128,128,8,8,8}, {128,128,16,8,8}
  };
  for (const Config& c : configs) {
    if (!legal(c)) continue;
    CUDA_CHECK(cudaMemset(dC, 0, bytes_c));
    if (c.bm == 64 && c.bk == 8 && c.tm == 4)
      record_result<64,64,8,4,4>(c, M, N, K, dA, dB, dC, repeats, csv);
    else if (c.bm == 64 && c.bk == 8)
      record_result<64,64,8,8,8>(c, M, N, K, dA, dB, dC, repeats, csv);
    else if (c.bm == 64)
      record_result<64,64,16,8,8>(c, M, N, K, dA, dB, dC, repeats, csv);
    else if (c.bn == 64 && c.bk == 8)
      record_result<128,64,8,8,4>(c, M, N, K, dA, dB, dC, repeats, csv);
    else if (c.bn == 64)
      record_result<128,64,16,8,4>(c, M, N, K, dA, dB, dC, repeats, csv);
    else if (c.bk == 8)
      record_result<128,128,8,8,8>(c, M, N, K, dA, dB, dC, repeats, csv);
    else
      record_result<128,128,16,8,8>(c, M, N, K, dA, dB, dC, repeats, csv);
  }
  CUDA_CHECK(cudaFree(dA));
  CUDA_CHECK(cudaFree(dB));
  CUDA_CHECK(cudaFree(dC));
}


In [ ]:
# Compile directly for the RTX 2080 Ti (compute capability 7.5).
# This emits sm_75 machine code and avoids driver-side PTX JIT incompatibility.
!nvcc -O3 -arch=sm_75 -Wno-deprecated-gpu-targets parameter_sensitivity.cu -o parameter_sensitivity

# Baseline square case.
!./parameter_sensitivity 2048 2048 2048 10
!cp parameter_sweep.csv parameter_sweep_2048.csv

# Non-square and non-power-of-two case.
!./parameter_sensitivity 3000 1500 2048 10
!cp parameter_sweep.csv parameter_sweep_3000x1500.csv

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

files = [Path("parameter_sweep_2048.csv"), Path("parameter_sweep_3000x1500.csv")]
frames = []
for file in files:
    if not file.exists() or file.stat().st_size == 0:
        continue
    frame = pd.read_csv(file)
    if frame.empty or "GFLOPS" not in frame.columns:
        continue
    frame["Shape"] = f"{frame.M.iloc[0]}x{frame.N.iloc[0]}x{frame.K.iloc[0]}"
    frame["Configuration"] = (
        "BM=" + frame.BM.astype(str) + ",BN=" + frame.BN.astype(str)
        + ",BK=" + frame.BK.astype(str) + ",TM=" + frame.TM.astype(str)
        + ",TN=" + frame.TN.astype(str)
    )
    frames.append(frame)

if not frames:
    raise RuntimeError("No valid sweep CSV files found. Run the CUDA compile/run cell successfully first.")

sweep = pd.concat(frames, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
for shape, group in sweep.groupby("Shape"):
    group = group.sort_values("GFLOPS")
    axes[0].plot(group["Configuration"], group["GFLOPS"], marker="o", label=shape)
    axes[1].plot(group["Configuration"], group["ElapsedMs"], marker="o", label=shape)

for axis, ylabel, title in [
    (axes[0], "GFLOP/s", "Throughput sensitivity"),
    (axes[1], "Milliseconds", "Runtime sensitivity"),
]:
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=75)
    axis.grid(True, linestyle="--", alpha=0.35)
    axis.legend(title="Shape")

plt.show()
sweep.sort_values(["Shape", "GFLOPS"], ascending=[True, False])[
    ["Shape", "Configuration", "Threads", "ElapsedMs", "GFLOPS"]
]

The measured results show that changing `BM`, `BN`, `BK`, `TM`, and `TN` changes performance substantially. On this RTX 2080 Ti, larger per-thread tiles can increase reuse, but configurations with 1024 threads and high register demand may fail with `too many resources requested for launch`. Therefore, invalid configurations must be excluded or analyzed separately as resource cliffs.

Among the configurations that successfully execute, report the fastest measured configuration for each matrix shape. Do not claim that `BK=32` is optimal unless a `BK=32` configuration has actually been benchmarked. The best configuration can change with GPU architecture, matrix dimensions, register allocation, shared-memory usage, and tile quantization.

For weird matrix dimensions (not power of 2) we can see that the performance of the kernel is not optimal (except one case) the reason for this is that kernel is generally optimized for power of 2 matrix dimensions. So for weird matrix some extra blocks are required which dont utlize the shared memory and hence the performance is not optimal. The Gflops/sec is less as a  result because of this.
